### Record Grouping

In [ ]:
"""
Solute Product Grouping Script
Processes textual data, computes cluster similarities, and uses MILP optimization 
to partition the dataset into optimal groups of 5, exporting to JSON.
"""

from __future__ import annotations

import json
import math
import re
from collections import Counter
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, Iterable, List, Tuple

import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csc_array, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

project_root = Path("").resolve()
os.chdir(project_root)

# ============================================================
# Parameters
# ============================================================
input_path = "./dataset/solute_products_unique.json"
output_json = "./dataset/solute-grouping.json"

neighbor_pool = 10
top_per_seed = 8
time_limit = 300
random_seed = 42

@dataclass(frozen=True)
class CandidateGroup:
    members: Tuple[int, ...]  # sorted cluster positions
    score: float
    origin: str


# ============================================================
# Text Processing & Summarization
# ============================================================
def normalize_text(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    text = str(value).lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def summarize_values(values: Iterable[object], top_k: int, max_len: int) -> List[str]:
    counter: Counter[str] = Counter()
    for value in values:
        text = normalize_text(value)
        if text:
            counter[text] += 1
    return [text[:max_len] for text, _ in counter.most_common(top_k)]


# ============================================================
# Table Building (Adapted for new JSON features)
# ============================================================
def build_cluster_table(df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for cluster_id, g in df.groupby("cluster_id", sort=True):
        
        # SKIP clusters with 1 or zero records
        if len(g) <= 1:
            continue

        # NOTE: Using 'name' and 'desc' instead of 'title' and 'description'
        brands = summarize_values(g.get("brand", []), top_k=3, max_len=60)
        names = summarize_values(g.get("name", []), top_k=6, max_len=180)
        descs = summarize_values(g.get("desc", []), top_k=2, max_len=240)

        rep_parts: List[str] = []
        rep_parts.extend([f"brand {x}" for x in brands])
        rep_parts.extend([f"name {x}" for x in names])
        rep_parts.extend([f"desc {x}" for x in descs])

        rows.append(
            {
                "cluster_id": int(cluster_id),
                "offer_count": int(len(g)),
                "brands": brands,
                "names": names,
                "descs": descs,
                "cluster_text": " ".join(rep_parts),
            }
        )

    cluster_df = pd.DataFrame(rows).reset_index(drop=True)
    cluster_df["cluster_pos"] = np.arange(len(cluster_df))
    return cluster_df


# ============================================================
# ML: Similarity Matrix & Grouping Logic
# ============================================================
def build_similarity_matrix(cluster_texts: List[str]) -> np.ndarray:
    word_vec = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), max_features=30000, sublinear_tf=True
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), max_features=40000, sublinear_tf=True
    )

    x_word = word_vec.fit_transform(cluster_texts)
    x_char = char_vec.fit_transform(cluster_texts)
    x = hstack([x_word, x_char]).tocsr()

    sim = linear_kernel(x, x)
    np.fill_diagonal(sim, 1.0)
    return np.asarray(sim, dtype=np.float64)

def group_score(sim: np.ndarray, members: Tuple[int, ...]) -> float:
    vals = [sim[a, b] for a, b in combinations(members, 2)]
    return float(np.mean(vals)) if vals else 0.0

def group_stats(sim: np.ndarray, members: Tuple[int, ...]) -> Tuple[float, float, float]:
    vals = [float(sim[a, b]) for a, b in combinations(members, 2)]
    if not vals:
        return 0.0, 0.0, 0.0
    return float(np.mean(vals)), float(np.min(vals)), float(np.max(vals))

def medoid_position(sim: np.ndarray, members: Tuple[int, ...]) -> int:
    best_pos = members[0]
    best_score = -1.0
    for p in members:
        s = 0.0
        for q in members:
            if p != q:
                s += float(sim[p, q])
        s /= max(1, len(members) - 1)
        if s > best_score:
            best_score = s
            best_pos = p
    return best_pos

def generate_topk_candidates(sim: np.ndarray, neighbor_pool: int = 10, top_per_seed: int = 8) -> List[CandidateGroup]:
    n = sim.shape[0]
    seen: Dict[Tuple[int, ...], CandidateGroup] = {}

    for seed in range(n):
        order = np.argsort(-sim[seed])
        neigh = [int(p) for p in order if int(p) != seed][:neighbor_pool]

        scored: List[Tuple[float, Tuple[int, ...]]] = []
        for combo in combinations(neigh, 4):
            members = tuple(sorted((seed, *combo)))
            scored.append((group_score(sim, members), members))

        scored.sort(reverse=True)
        for score, members in scored[:top_per_seed]:
            prev = seen.get(members)
            cand = CandidateGroup(members=members, score=score, origin="topk")
            if prev is None or cand.score > prev.score:
                seen[members] = cand
    return list(seen.values())

def greedy_partition(sim: np.ndarray, positions: List[int], strategy: str, rng: np.random.Generator) -> List[CandidateGroup]:
    remaining = set(int(p) for p in positions)
    groups: List[CandidateGroup] = []

    def pick_seed() -> int:
        rem = sorted(remaining)
        if strategy == "random":
            return int(rng.choice(rem))
        if strategy == "avg_local":
            best_seed = rem[0]
            best_val = -1.0
            for p in rem:
                others = [q for q in rem if q != p]
                if not others: continue
                top = sorted((sim[p, q] for q in others), reverse=True)[:8]
                val = float(np.mean(top)) if top else -1.0
                if val > best_val:
                    best_val = val
                    best_seed = p
            return best_seed
        return max(rem, key=lambda p: float(np.mean([sim[p, q] for q in rem if q != p][:20] or [0.0])))

    while len(remaining) >= 5:
        seed = pick_seed()
        rem_others = [q for q in remaining if q != seed]
        rem_others.sort(key=lambda q: sim[seed, q], reverse=True)

        chosen = tuple(sorted((seed, *rem_others[:4])))
        groups.append(CandidateGroup(members=chosen, score=group_score(sim, chosen), origin=f"partition_{strategy}"))
        for p in chosen:
            remaining.remove(p)
    return groups

def build_feasible_partition_candidates(sim: np.ndarray, seed: int = 42) -> List[CandidateGroup]:
    rng = np.random.default_rng(seed)
    positions = list(range(sim.shape[0]))
    cands: Dict[Tuple[int, ...], CandidateGroup] = {}

    for strategy in ["avg_local", "degree", "random", "random", "random"]:
        groups = greedy_partition(sim, positions, strategy, rng)
        for cand in groups:
            prev = cands.get(cand.members)
            if prev is None or cand.score > prev.score:
                cands[cand.members] = cand
    return list(cands.values())

def dedupe_candidates(cands: List[CandidateGroup]) -> List[CandidateGroup]:
    best: Dict[Tuple[int, ...], CandidateGroup] = {}
    for c in cands:
        prev = best.get(c.members)
        if prev is None or c.score > prev.score:
            best[c.members] = c
    return list(best.values())


# ============================================================
# Optimization (Scipy MILP)
# ============================================================
def solve_exact_group_selection(cands: List[CandidateGroup], cluster_count: int, target_groups: int = 160, time_limit: int = 300):
    m = len(cands)
    row_idx, col_idx, data = [], [], []

    for j, cand in enumerate(cands):
        for p in cand.members:
            row_idx.append(p)
            col_idx.append(j)
            data.append(1.0)

    A1 = csc_array((data, (row_idx, col_idx)), shape=(cluster_count, m))
    b_l1 = np.zeros(cluster_count)
    b_u1 = np.ones(cluster_count)

    A2 = csc_array(np.ones((1, m), dtype=float))
    b_l2 = np.array([float(target_groups)])
    b_u2 = np.array([float(target_groups)])

    A = csc_array(np.vstack([A1.toarray(), A2.toarray()]))
    constraints = LinearConstraint(A, np.concatenate([b_l1, b_l2]), np.concatenate([b_u1, b_u2]))
    bounds = Bounds(np.zeros(m), np.ones(m))
    integrality = np.ones(m, dtype=int)
    c = -np.array([cand.score for cand in cands], dtype=float)

    return milp(
        c=c,
        constraints=constraints,
        integrality=integrality,
        bounds=bounds,
        options={"time_limit": float(time_limit), "presolve": True, "mip_rel_gap": 0.0},
    )


# ============================================================
# JSON Export (Adapted for new JSON features)
# ============================================================
def build_output_json(selected: List[CandidateGroup], cluster_df: pd.DataFrame, sim: np.ndarray) -> Dict[str, object]:
    selected_sorted = sorted(selected, key=lambda g: g.score, reverse=True)
    groups_json: List[Dict[str, object]] = []
    used_positions: set[int] = set()

    for rank, group in enumerate(selected_sorted, start=1):
        gid = f"G{rank:03d}"
        mean_sim, min_sim, max_sim = group_stats(sim, group.members)
        seed_pos = medoid_position(sim, group.members)
        seed_cluster_id = int(cluster_df.loc[seed_pos, "cluster_id"])

        ordered_members = [seed_pos] + [p for p in group.members if p != seed_pos]
        group_members_json: List[Dict[str, object]] = []

        for member_order, pos in enumerate(ordered_members, start=1):
            used_positions.add(pos)
            cluster_id = int(cluster_df.loc[pos, "cluster_id"])
            sim_to_seed = 1.0 if pos == seed_pos else float(sim[seed_pos, pos])

            group_members_json.append({
                "cluster_id": cluster_id,
                "member_order_in_group": member_order,
                "is_seed": bool(pos == seed_pos),
                "similarity_to_seed": sim_to_seed,
                "offer_count": int(cluster_df.loc[pos, "offer_count"]),
                "brands": cluster_df.loc[pos, "brands"],
                "names": cluster_df.loc[pos, "names"], # Adapted feature
            })

        groups_json.append({
            "group_id": gid,
            "group_rank_desc": rank,
            "group_origin": group.origin,
            "seed_cluster_id": seed_cluster_id,
            "group_size": 5,
            "mean_pairwise_similarity": mean_sim,
            "min_pairwise_similarity": min_sim,
            "max_pairwise_similarity": max_sim,
            "clusters": group_members_json,
        })

    leftover_positions = [int(p) for p in cluster_df["cluster_pos"].tolist() if int(p) not in used_positions]
    leftovers_json: List[Dict[str, object]] = []
    
    for idx, pos in enumerate(leftover_positions, start=1):
        leftovers_json.append({
            "leftover_rank": idx,
            "cluster_id": int(cluster_df.loc[pos, "cluster_id"]),
            "offer_count": int(cluster_df.loc[pos, "offer_count"]),
            "brands": cluster_df.loc[pos, "brands"],
            "names": cluster_df.loc[pos, "names"], # Adapted feature
        })

    return {
        "summary": {
            "total_clusters": int(len(cluster_df)),
            "group_count": int(len(groups_json)),
            "clusters_in_groups": int(len(groups_json) * 5),
            "leftover_count": int(len(leftovers_json)),
        },
        "groups": groups_json,
        "leftovers": leftovers_json,
    }


# ============================================================
# Main Execution Block
# ============================================================
if __name__ == "__main__":
    print(f"Loading data from {input_path}...")
    with open(input_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    df = pd.DataFrame(raw)
    print("Columns:", list(df.columns))
    print("Rows:", len(df))
    print("Unique cluster_id:", df["cluster_id"].nunique())

    print("\nBuilding cluster table...")
    cluster_df = build_cluster_table(df)
    print("Cluster table size (after dropping size <= 1):", cluster_df.shape)

    print("\nCalculating similarities...")
    sim = build_similarity_matrix(cluster_df["cluster_text"].tolist())
    print("Similarity matrix shape:", sim.shape)

    print("\nGenerating candidates...")
    topk_cands = generate_topk_candidates(sim, neighbor_pool=neighbor_pool, top_per_seed=top_per_seed)
    partition_cands = build_feasible_partition_candidates(sim, seed=random_seed)
    candidates = dedupe_candidates(topk_cands + partition_cands)

    print("Top-k candidates:", len(topk_cands))
    print("Partition candidates:", len(partition_cands))
    print("Deduplicated candidates:", len(candidates))

    # SAFEGUARD: Ensure target_groups doesn't exceed the mathematical limit
    max_possible_groups = len(cluster_df) // 5
    desired_target_groups = 160
    actual_target_groups = min(desired_target_groups, max_possible_groups)

    print(f"\nSolving Exact Group Selection (Time limit: {time_limit}s)...")
    print(f"Targeting {actual_target_groups} groups out of a maximum possible {max_possible_groups}.")
    
    res = solve_exact_group_selection(
        candidates,
        cluster_count=len(cluster_df),
        target_groups=actual_target_groups, 
        time_limit=time_limit,
    )

    print("Success:", getattr(res, "success", False))
    print("Status:", getattr(res, "status", None))
    print("Message:", getattr(res, "message", ""))

    if not getattr(res, "success", False):
        raise RuntimeError(f"MILP did not solve successfully. status={res.status}, message={res.message}")

    x = np.rint(res.x).astype(int)
    selected = [cand for cand, flag in zip(candidates, x) if flag == 1]
    print("Selected groups:", len(selected))

    output = build_output_json(selected, cluster_df, sim)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\nSuccess! Wrote output to: {output_json}")
    print(output["summary"])

### Sample Generation

In [ ]:
"""
Solute Sample Generation Script
Handles extracting subsets from the grouped data via round-robin size bucketing
and compiling them into a final sample dataset format.
"""

import json
import random
from pathlib import Path
from collections import defaultdict
import pandas as pd

# -----------------------
# Configuration
# -----------------------
# Input files
ORIGINAL_DATA_PATH = "./dataset/solute_products_unique.json"
RECOVERED_JSON_PATH = "./dataset/solute-grouping.json"

# Output files
OUTPUT_SAMPLE_CSV = "./dataset/sample_solute.csv"
OUTPUT_MANIFEST_JSON = "./dataset/solute_manifest.json"

# Sampling rules
RANDOM_SEED = 4
GROUPS_TO_SAMPLE = 16
EXTRA_CLUSTERS_TO_SAMPLE = 20

# Set seed for reproducible sampling
random.seed(RANDOM_SEED)

# -----------------------
# 1. Load original flat dataset & Compute Sizes
# -----------------------
print(f"Loading original dataset from {ORIGINAL_DATA_PATH}...")
with open(ORIGINAL_DATA_PATH, "r", encoding="utf-8") as f:
    original_records = json.load(f)

original_df = pd.DataFrame(original_records)
original_columns = original_df.columns.tolist()

# Pre-compute all cluster sizes globally
cluster_sizes = original_df.groupby("cluster_id").size().to_dict()

print("Original columns:", original_columns)
print("Total records:", len(original_df))
print("Unique clusters:", original_df["cluster_id"].nunique())


# -----------------------
# 2. Load recovered groups
# -----------------------
print(f"\nLoading groups from {RECOVERED_JSON_PATH}...")
with open(RECOVERED_JSON_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

all_groups = []
for g in payload["groups"]:
    cluster_ids = [int(x["cluster_id"]) for x in g["clusters"]]
    # Calculate the total sum of records for the 5 clusters in this group
    total_records = sum(cluster_sizes.get(cid, 0) for cid in cluster_ids)
    
    all_groups.append({
        "group_id": g["group_id"],
        "group_rank_desc": int(g["group_rank_desc"]),
        "cluster_ids": cluster_ids,
        "total_records": total_records
    })

print("Recovered groups available:", len(all_groups))


# -----------------------
# 3. Select Groups (Equal Size Round-Robin)
# -----------------------
# Bucket the groups based on their total record counts
group_size_buckets = defaultdict(list)
for g in all_groups:
    group_size_buckets[g["total_records"]].append(g)

# Shuffle groups inside each bucket
for size in group_size_buckets:
    random.shuffle(group_size_buckets[size])

available_group_sizes = sorted(group_size_buckets.keys())
selected_groups = []

# Round-robin loop for groups
g_size_index = 0
while len(selected_groups) < GROUPS_TO_SAMPLE and available_group_sizes:
    current_size = available_group_sizes[g_size_index % len(available_group_sizes)]
    
    if group_size_buckets[current_size]:
        selected_groups.append(group_size_buckets[current_size].pop())
        g_size_index += 1
    else:
        available_group_sizes.remove(current_size)
        if available_group_sizes:
            g_size_index = g_size_index % len(available_group_sizes)

assert len(selected_groups) == GROUPS_TO_SAMPLE, f"Expected at least {GROUPS_TO_SAMPLE} groups available."

# Finalize sample groups processing
random.shuffle(selected_groups)
sample_group_cluster_ids = sorted({cid for g in selected_groups for cid in g["cluster_ids"]})
sample_group_sizes = sorted([g["total_records"] for g in selected_groups])

print("\n--- Group Selection Distribution (Total records per group) ---")
print(f"Sample Group Sizes (sorted): {sample_group_sizes}")
print(f"Sample grouped clusters count:", len(sample_group_cluster_ids))


# -----------------------
# 4. Build remaining pool for extra clusters (Equal Size Round-Robin)
# -----------------------
# Prevent overlap with already assigned grouped clusters
selected_group_cluster_ids = {cid for g in selected_groups for cid in g["cluster_ids"]}

# Get valid remaining pool (clusters not already used, and MUST have size > 1)
valid_remaining = {cid for cid, size in cluster_sizes.items() if cid not in selected_group_cluster_ids and size > 1}

assert len(valid_remaining) >= EXTRA_CLUSTERS_TO_SAMPLE, "Not enough remaining clusters."

# Group valid remaining clusters strictly by their size
size_buckets = defaultdict(list)
for cid in valid_remaining:
    size_buckets[cluster_sizes[cid]].append(cid)

# Shuffle inside each bucket so we pick random clusters of that size
for size in size_buckets:
    random.shuffle(size_buckets[size])

available_cluster_sizes = sorted(size_buckets.keys())
extra_clusters = []

# Round-robin loop for extra clusters
c_size_index = 0
while len(extra_clusters) < EXTRA_CLUSTERS_TO_SAMPLE and available_cluster_sizes:
    current_size = available_cluster_sizes[c_size_index % len(available_cluster_sizes)]
    
    if size_buckets[current_size]:
        extra_clusters.append(size_buckets[current_size].pop())
        c_size_index += 1
    else:
        available_cluster_sizes.remove(current_size)
        if available_cluster_sizes:
            c_size_index = c_size_index % len(available_cluster_sizes)

# Finalize extra clusters processing
sample_extra_clusters = sorted(extra_clusters)
sample_extra_sizes = sorted([cluster_sizes.get(cid, 0) for cid in sample_extra_clusters])

print("\n--- Extra Clusters Distribution (Records per cluster) ---")
print("Remaining pool size (for extra clusters > size 1):", len(valid_remaining))
print(f"Sample extra cluster sizes (sorted): {sample_extra_sizes}")


# -----------------------
# 5. Materialize flat sample dataset
# -----------------------
sample_cluster_ids = set(sample_group_cluster_ids) | set(sample_extra_clusters)
sample_df = original_df[original_df["cluster_id"].astype(int).isin(sample_cluster_ids)].copy()

# Enforce original column order
sample_df = sample_df[original_columns]
sample_df.to_csv(OUTPUT_SAMPLE_CSV, index=False)

print("\n--- Export Results ---")
print(f"Wrote: {OUTPUT_SAMPLE_CSV}")
print(f"Sample - Rows: {len(sample_df)} | Unique clusters: {sample_df['cluster_id'].nunique()}")


# -----------------------
# 6. Save a manifest for traceability
# -----------------------
manifest = {
    "recovered_source": str(RECOVERED_JSON_PATH),
    "random_seed": RANDOM_SEED,
    "rules": {
        "groups_sampled_method": "equal_size_round_robin",
        "groups_to_sample": GROUPS_TO_SAMPLE,
        "extra_clusters_to_sample": EXTRA_CLUSTERS_TO_SAMPLE,
        "extra_clusters_sampling_method": "equal_size_round_robin",
        "output_format": "csv",
        "output_columns_match_original": True,
    },
    "sample": {
        "group_ids": [g["group_id"] for g in selected_groups],
        "group_sizes_total_records": sample_group_sizes,
        "group_cluster_ids": sample_group_cluster_ids,
        "extra_cluster_ids": sample_extra_clusters,
        "extra_cluster_sizes": sample_extra_sizes,  
        "all_cluster_ids": sorted(sample_cluster_ids),
        "row_count": int(len(sample_df)),
    }
}

with open(OUTPUT_MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"\nWrote tracking manifest: {OUTPUT_MANIFEST_JSON}")

### Ground Truth Generation

In [ ]:
"""
Solute Ground Truth Generation Script
Generates consecutive adjacent matching pairs for records sharing a cluster ID.
Produces the definitive ground truth CSV.
"""

import pandas as pd
from pathlib import Path

# ----------------------------
# Paths
# ----------------------------
def resolve_path(path_str: str) -> Path:
    """Resolve a path relative to the current working directory or its parent."""
    path = Path(path_str)
    if path.is_absolute():
        return path

    cwd_candidate = Path.cwd() / path
    parent_candidate = Path.cwd().parent / path

    if cwd_candidate.exists():
        return cwd_candidate
    if parent_candidate.exists():
        return parent_candidate
    return cwd_candidate


SAMPLE_CSV_PATH = resolve_path("./dataset/sample_solute.csv")
OUTPUT_GT_CSV = resolve_path("./dataset/sample_solute_gt.csv")

# ----------------------------
# Core Functionality
# ----------------------------
def generate_ground_truth_adjacent_pairs(sample_csv_path: Path, output_csv_path: Path) -> pd.DataFrame:
    """
    Reads the sample dataset and pairs adjacent rows within each cluster 
    as positive matching instances.

    Expected columns in output: ['ltable_id', 'rtable_id']
    """
    print(f"Loading sampled data from {sample_csv_path}...")
    df = pd.read_csv(sample_csv_path)

    # Validate essential columns exist
    required_cols = {"id", "cluster_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in dataset: {missing}")

    # Track original row order natively before groupby breaks it
    df = df.reset_index(drop=False).rename(columns={"index": "_row_order"})

    gt_rows = []

    # Iterate over unique clusters
    for cluster_id, group in df.groupby("cluster_id", sort=False):
        group = group.sort_values("_row_order")
        ids = group["id"].tolist()

        # Connect elements sequentially (n-1 pairs for n cluster items)
        if len(ids) >= 2:
            for i in range(len(ids) - 1):
                gt_rows.append({
                    "ltable_id": ids[i],
                    "rtable_id": ids[i + 1]
                })

    # Materialize dataframe and write output
    gt_df = pd.DataFrame(gt_rows, columns=["ltable_id", "rtable_id"])
    gt_df.to_csv(output_csv_path, index=False)

    print(f"Saved Ground Truth File: {output_csv_path}")
    print(f"Total positive matching pairs: {len(gt_df)}")

    return gt_df

# ----------------------------
# Execution
# ----------------------------
if __name__ == "__main__":
    if SAMPLE_CSV_PATH.exists():
        gt = generate_ground_truth_adjacent_pairs(SAMPLE_CSV_PATH, OUTPUT_GT_CSV)

        print("\nGround Truth Preview:")
        print(gt.head())
    else:
        print(f"Error: Could not locate '{SAMPLE_CSV_PATH}'. Run the sampling script first.")